# Subastas Casanare - Notebook completo

Version: 1.1.0

Datos en Google Drive: **`MyDrive/subasta_ganadera_data/`**
- `data/lotes/` — detalle por lote
- `data/resumen/` — resultados generales por edad/sexo
- `data/combined/` — Parquet consolidado
- `pdfs/` — PDFs descargados

Flujo rapido: ejecuta la **celda unica** (montar Drive + descargar ferias).
O el flujo completo: instalacion → Drive → extraccion → CSV → graficos.

In [ ]:
# Celda 1 - Instalar dependencias
!pip install -q pandas pdfplumber requests pyarrow plotly openpyxl

In [ ]:
# Celda 2 - Montar Drive (subasta_ganadera_data)
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PROYECTO = Path("/content/drive/MyDrive/subasta_ganadera_data")

DATA_DIR = PROYECTO / "data"
LOTE_DIR = DATA_DIR / "lotes"
RESUMEN_DIR = DATA_DIR / "resumen"
COMBINED_DIR = DATA_DIR / "combined"
EXPORT_DIR = PROYECTO / "export"
PDF_DIR = PROYECTO / "pdfs"

for d in [PROYECTO, LOTE_DIR, RESUMEN_DIR, COMBINED_DIR, EXPORT_DIR, PDF_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Proyecto:", PROYECTO)
print("Carpetas: lotes | resumen | combined | pdfs | export")

## Celda unica (recomendada)

Ejecuta **solo esta celda** para montar Drive, descargar ferias nuevas y guardar en `lotes/` y `resumen/`. Omite ferias ya procesadas.

In [ ]:
# === CELDA UNICA: montar Drive + descargar ferias ===
!pip install -q pandas pdfplumber requests pyarrow

import re
from pathlib import Path

import pandas as pd
import pdfplumber
import requests
from google.colab import drive

# --- Configuracion ---
FERIA_DESDE = 1950
FERIA_HASTA = 1960

drive.mount("/content/drive")

PROYECTO = Path("/content/drive/MyDrive/subasta_ganadera_data")
LOTE_DIR = PROYECTO / "data" / "lotes"
RESUMEN_DIR = PROYECTO / "data" / "resumen"
COMBINED_DIR = PROYECTO / "data" / "combined"
PDF_DIR = PROYECTO / "pdfs"
for d in [LOTE_DIR, RESUMEN_DIR, COMBINED_DIR, PDF_DIR]:
    d.mkdir(parents=True, exist_ok=True)

ROW_PATTERN = re.compile(
    r"^(\d+)\s+([A-Z]{2})\s+(\d+)\s+([\d.]+)\s+([\d.]+)\s+"
    r"(.+?)\s+(\d{2}:\d{2}:\d{2})\s+([\d.]+)\s+([\d.]+)(?:\s+(.*))?$"
)
SUMMARY_ROW_PATTERN = re.compile(
    r"^(.+?)\s+([A-Z]{2})\s+(\d+)\s+"
    r"([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)$"
)
HEADER_PATTERN = re.compile(
    r"FERIA NO\.\s*(\d+).*?CIUDAD\s+(.+?)\s+FECHA FERIA\.\s*(\d{4}-\d{2}-\d{2})", re.DOTALL
)
SECTION_PATTERN = re.compile(r"^(REMATE|SUBASTA)\s+(.+)$")
LOTES_SKIP = ("SUBASTA", "LISTADO", "FERIA", "FECHA", "Lote Sexo", "RESULTADOS", "TOTALES:", "REMATE")
RESUMEN_SKIP = ("SUBASTA GANADERA", "LISTADO", "FERIA", "FECHA", "Lote Sexo", "RESULTADOS GENERALES", "Edad Sexo", "TOTALES:")


def parse_num(v):
    return int(str(v).replace(".", ""))


def download_pdf(feria: int) -> Path:
    url = f"https://www.subacasanare.com/Precio_Pdf/{feria}"
    dest = PDF_DIR / f"{feria}.pdf"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    if not r.content.startswith(b"%PDF"):
        raise ValueError("respuesta no es PDF")
    dest.write_bytes(r.content)
    return dest


def extract_metadata(text: str) -> dict:
    m = HEADER_PATTERN.search(text)
    if not m:
        return {}
    return {"feria_no": m.group(1), "ciudad": m.group(2).strip(), "fecha_subasta": m.group(3)}


def extract_lotes(pdf_path: Path) -> pd.DataFrame:
    records, current, in_summary = [], None, False
    with pdfplumber.open(pdf_path) as pdf:
        meta = extract_metadata(pdf.pages[0].extract_text() or "")
        for page_num, page in enumerate(pdf.pages, 1):
            for line in (page.extract_text() or "").split("\n"):
                line = line.strip()
                if "RESULTADOS GENERALES" in line:
                    in_summary = True
                if not line or in_summary or line.startswith(LOTES_SKIP):
                    continue
                match = ROW_PATTERN.match(line)
                if match:
                    if current:
                        records.append(current)
                    current = {
                        "pagina": page_num, "lote": int(match.group(1)), "sexo": match.group(2),
                        "cantidad": int(match.group(3)), "peso_total": parse_num(match.group(4)),
                        "peso_promedio": parse_num(match.group(5)), "procedencia": match.group(6).strip(),
                        "entrada": match.group(7), "precio_base": parse_num(match.group(8)),
                        "precio_final": parse_num(match.group(9)), "observaciones": (match.group(10) or "").strip(),
                    }
                elif current:
                    current["observaciones"] = (current["observaciones"] + " " + line).strip()
    if current:
        records.append(current)
    df = pd.DataFrame(records)
    for k, v in meta.items():
        df[k] = v
    return df


def extract_resumen(pdf_path: Path) -> pd.DataFrame:
    records, current_section, in_summary = [], None, False
    with pdfplumber.open(pdf_path) as pdf:
        meta = extract_metadata(pdf.pages[0].extract_text() or "")
        for page_num, page in enumerate(pdf.pages, 1):
            for line in (page.extract_text() or "").split("\n"):
                line = line.strip()
                if not line:
                    continue
                if "RESULTADOS GENERALES" in line:
                    in_summary = True
                    continue
                if not in_summary or line.startswith(RESUMEN_SKIP):
                    continue
                sec = SECTION_PATTERN.match(line)
                if sec:
                    current_section = f"{sec.group(1)} {sec.group(2).strip()}"
                    continue
                match = SUMMARY_ROW_PATTERN.match(line)
                if match:
                    records.append({
                        "pagina": page_num, "seccion": current_section,
                        "edad": match.group(1).strip(), "sexo": match.group(2),
                        "cantidad": int(match.group(3)), "peso_med": parse_num(match.group(4)),
                        "minimo": parse_num(match.group(5)), "maximo": parse_num(match.group(6)),
                        "promedio": parse_num(match.group(7)), "vr_animal": parse_num(match.group(8)),
                    })
    df = pd.DataFrame(records)
    for k, v in meta.items():
        df[k] = v
    return df


def feria_procesada(feria: int) -> bool:
    p = LOTE_DIR / f"feria_{feria}.parquet"
    if not p.exists() or p.stat().st_size == 0:
        return False
    try:
        df = pd.read_parquet(p)
        return not df.empty
    except Exception:
        return False


def guardar_feria(lotes_df: pd.DataFrame, resumen_df: pd.DataFrame, feria: int):
    if "fecha_subasta" in lotes_df.columns:
        lotes_df = lotes_df.copy()
        lotes_df["fecha_subasta"] = pd.to_datetime(lotes_df["fecha_subasta"], errors="coerce")
    lotes_df.to_parquet(LOTE_DIR / f"feria_{feria}.parquet", index=False)
    if not resumen_df.empty:
        if "fecha_subasta" in resumen_df.columns:
            resumen_df = resumen_df.copy()
            resumen_df["fecha_subasta"] = pd.to_datetime(resumen_df["fecha_subasta"], errors="coerce")
        resumen_df.to_parquet(RESUMEN_DIR / f"feria_{feria}.parquet", index=False)


def rebuild_combined():
    lotes_files = sorted(LOTE_DIR.glob("feria_*.parquet"))
    resumen_files = sorted(RESUMEN_DIR.glob("feria_*.parquet"))
    lotes = pd.concat([pd.read_parquet(f) for f in lotes_files], ignore_index=True) if lotes_files else pd.DataFrame()
    resumen = pd.concat([pd.read_parquet(f) for f in resumen_files], ignore_index=True) if resumen_files else pd.DataFrame()
    if not lotes.empty:
        lotes.to_parquet(COMBINED_DIR / "lotes.parquet", index=False)
    if not resumen.empty:
        resumen.to_parquet(COMBINED_DIR / "resumen.parquet", index=False)
    return lotes, resumen


ok = skip = fail = 0
for feria in range(FERIA_DESDE, FERIA_HASTA + 1):
    if feria_procesada(feria):
        print(f"[SKIP] {feria}")
        skip += 1
        continue
    try:
        pdf = download_pdf(feria)
        lotes_df = extract_lotes(pdf)
        resumen_df = extract_resumen(pdf)
        if lotes_df.empty:
            raise ValueError("sin lotes")
        guardar_feria(lotes_df, resumen_df, feria)
        print(f"[OK] {feria} - {len(lotes_df)} lotes, {len(resumen_df)} resumen")
        ok += 1
    except Exception as e:
        print(f"[FAIL] {feria}: {e}")
        fail += 1

lotes, resumen = rebuild_combined()
print(f"\nResultado: OK={ok} SKIP={skip} FAIL={fail}")
print(f"Combined: {len(lotes)} lotes | {len(resumen)} filas resumen")
print(f"Guardado en: {PROYECTO}")

## 3. Ingesta de ferias

Descarga PDFs de Subacasanare, extrae lotes y guarda Parquet.
Las ferias ya procesadas se omiten automaticamente.

In [ ]:
# Celda 4 - Codigo de extraccion (lotes + resumen)
import re
import requests
import pdfplumber
import pandas as pd
from pathlib import Path

ROW_PATTERN = re.compile(
    r"^(\d+)\s+([A-Z]{2})\s+(\d+)\s+([\d.]+)\s+([\d.]+)\s+"
    r"(.+?)\s+(\d{2}:\d{2}:\d{2})\s+([\d.]+)\s+([\d.]+)(?:\s+(.*))?$"
)
SUMMARY_ROW_PATTERN = re.compile(
    r"^(.+?)\s+([A-Z]{2})\s+(\d+)\s+"
    r"([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)$"
)
HEADER_PATTERN = re.compile(
    r"FERIA NO\.\s*(\d+).*?CIUDAD\s+(.+?)\s+FECHA FERIA\.\s*(\d{4}-\d{2}-\d{2})", re.DOTALL
)
SECTION_PATTERN = re.compile(r"^(REMATE|SUBASTA)\s+(.+)$")
LOTES_SKIP = ("SUBASTA", "LISTADO", "FERIA", "FECHA", "Lote Sexo", "RESULTADOS", "TOTALES:", "REMATE")
RESUMEN_SKIP = ("SUBASTA GANADERA", "LISTADO", "FERIA", "FECHA", "Lote Sexo", "RESULTADOS GENERALES", "Edad Sexo", "TOTALES:")

def parse_num(v):
    return int(str(v).replace(".", ""))

def download_pdf(feria: int, dest_dir: Path) -> Path:
    url = f"https://www.subacasanare.com/Precio_Pdf/{feria}"
    dest = dest_dir / f"{feria}.pdf"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    if not r.content.startswith(b"%PDF"):
        raise ValueError(f"Feria {feria}: respuesta no es PDF")
    dest.write_bytes(r.content)
    return dest

def extract_metadata(text: str) -> dict:
    m = HEADER_PATTERN.search(text)
    if not m:
        return {}
    return {"feria_no": m.group(1), "ciudad": m.group(2).strip(), "fecha_subasta": m.group(3)}

def extract_lotes(pdf_path: Path) -> pd.DataFrame:
    records, current, in_summary = [], None, False
    with pdfplumber.open(pdf_path) as pdf:
        meta = extract_metadata(pdf.pages[0].extract_text() or "")
        for page_num, page in enumerate(pdf.pages, 1):
            for line in (page.extract_text() or "").split("\n"):
                line = line.strip()
                if "RESULTADOS GENERALES" in line:
                    in_summary = True
                if not line or in_summary or line.startswith(LOTES_SKIP):
                    continue
                match = ROW_PATTERN.match(line)
                if match:
                    if current:
                        records.append(current)
                    current = {
                        "pagina": page_num, "lote": int(match.group(1)), "sexo": match.group(2),
                        "cantidad": int(match.group(3)), "peso_total": parse_num(match.group(4)),
                        "peso_promedio": parse_num(match.group(5)), "procedencia": match.group(6).strip(),
                        "entrada": match.group(7), "precio_base": parse_num(match.group(8)),
                        "precio_final": parse_num(match.group(9)), "observaciones": (match.group(10) or "").strip(),
                    }
                elif current:
                    current["observaciones"] = (current["observaciones"] + " " + line).strip()
    if current:
        records.append(current)
    df = pd.DataFrame(records)
    for k, v in meta.items():
        df[k] = v
    return df

def extract_resumen(pdf_path: Path) -> pd.DataFrame:
    records, current_section, in_summary = [], None, False
    with pdfplumber.open(pdf_path) as pdf:
        meta = extract_metadata(pdf.pages[0].extract_text() or "")
        for page_num, page in enumerate(pdf.pages, 1):
            for line in (page.extract_text() or "").split("\n"):
                line = line.strip()
                if not line:
                    continue
                if "RESULTADOS GENERALES" in line:
                    in_summary = True
                    continue
                if not in_summary or line.startswith(RESUMEN_SKIP):
                    continue
                sec = SECTION_PATTERN.match(line)
                if sec:
                    current_section = f"{sec.group(1)} {sec.group(2).strip()}"
                    continue
                match = SUMMARY_ROW_PATTERN.match(line)
                if match:
                    records.append({
                        "pagina": page_num, "seccion": current_section,
                        "edad": match.group(1).strip(), "sexo": match.group(2),
                        "cantidad": int(match.group(3)), "peso_med": parse_num(match.group(4)),
                        "minimo": parse_num(match.group(5)), "maximo": parse_num(match.group(6)),
                        "promedio": parse_num(match.group(7)), "vr_animal": parse_num(match.group(8)),
                    })
    df = pd.DataFrame(records)
    for k, v in meta.items():
        df[k] = v
    return df

def feria_procesada(feria: int) -> bool:
    p = LOTE_DIR / f"feria_{feria}.parquet"
    if not p.exists() or p.stat().st_size == 0:
        return False
    try:
        return not pd.read_parquet(p).empty
    except Exception:
        return False

def guardar_feria(lotes_df: pd.DataFrame, resumen_df: pd.DataFrame, feria: int):
    lotes_df = lotes_df.copy()
    lotes_df["fecha_subasta"] = pd.to_datetime(lotes_df["fecha_subasta"], errors="coerce")
    lotes_df.to_parquet(LOTE_DIR / f"feria_{feria}.parquet", index=False)
    if not resumen_df.empty:
        resumen_df = resumen_df.copy()
        resumen_df["fecha_subasta"] = pd.to_datetime(resumen_df["fecha_subasta"], errors="coerce")
        resumen_df.to_parquet(RESUMEN_DIR / f"feria_{feria}.parquet", index=False)

def rebuild_combined():
    lotes_files = sorted(LOTE_DIR.glob("feria_*.parquet"))
    resumen_files = sorted(RESUMEN_DIR.glob("feria_*.parquet"))
    lotes = pd.concat([pd.read_parquet(f) for f in lotes_files], ignore_index=True) if lotes_files else pd.DataFrame()
    resumen = pd.concat([pd.read_parquet(f) for f in resumen_files], ignore_index=True) if resumen_files else pd.DataFrame()
    if not lotes.empty:
        lotes.to_parquet(COMBINED_DIR / "lotes.parquet", index=False)
    if not resumen.empty:
        resumen.to_parquet(COMBINED_DIR / "resumen.parquet", index=False)
    return lotes, resumen

print("Funciones de extraccion listas (lotes + resumen).")

In [ ]:
# Celda 5 - Descargar rango de ferias
FERIA_DESDE = 1950
FERIA_HASTA = 1960

ok = skip = fail = 0
for feria in range(FERIA_DESDE, FERIA_HASTA + 1):
    if feria_procesada(feria):
        print(f"[SKIP] {feria}")
        skip += 1
        continue
    try:
        pdf = download_pdf(feria, PDF_DIR)
        lotes_df = extract_lotes(pdf)
        resumen_df = extract_resumen(pdf)
        if lotes_df.empty:
            raise ValueError("sin lotes")
        guardar_feria(lotes_df, resumen_df, feria)
        print(f"[OK] {feria} - {len(lotes_df)} lotes, {len(resumen_df)} resumen")
        ok += 1
    except Exception as e:
        print(f"[FAIL] {feria}: {e}")
        fail += 1

lotes, resumen = rebuild_combined()
print(f"\nResultado: OK={ok} SKIP={skip} FAIL={fail}")
print(f"Combined: {len(lotes)} lotes | {len(resumen)} filas resumen")

## 4. Exportar CSV (Looker Studio)

In [ ]:
# Celda 6 - Parquet a CSV
lotes = pd.read_parquet(COMBINED_DIR / "lotes.parquet")
lotes_csv = EXPORT_DIR / "lotes_consolidado.csv"
lotes.to_csv(lotes_csv, index=False, encoding="utf-8-sig")
print(f"CSV lotes: {lotes_csv} ({len(lotes)} filas)")

resumen_path = COMBINED_DIR / "resumen.parquet"
if resumen_path.exists():
    resumen = pd.read_parquet(resumen_path)
    resumen_csv = EXPORT_DIR / "resumen_consolidado.csv"
    resumen.to_csv(resumen_csv, index=False, encoding="utf-8-sig")
    print(f"CSV resumen: {resumen_csv} ({len(resumen)} filas)")

lotes.head()

## 5. Graficos - Precio por sexo con variacion %

In [ ]:
# Celda 7 - Analisis precio por sexo
import plotly.graph_objects as go
from plotly.subplots import make_subplots

MESES = {1:"Ene",2:"Feb",3:"Mar",4:"Abr",5:"May",6:"Jun",7:"Jul",8:"Ago",9:"Sep",10:"Oct",11:"Nov",12:"Dic"}

df = lotes.copy()
df["fecha_subasta"] = pd.to_datetime(df["fecha_subasta"])
df["anio"] = df["fecha_subasta"].dt.year
df["mes"] = df["fecha_subasta"].dt.month
df["mes_nombre"] = df["mes"].map(MESES)

summary = (
    df.groupby(["sexo","anio","mes","mes_nombre"], as_index=False)
    .agg(precio_promedio=("precio_final","mean"), lotes=("lote","count"))
    .sort_values(["sexo","anio","mes"])
)
summary["periodo"] = summary["anio"].astype(str) + "-" + summary["mes"].astype(str).str.zfill(2)
summary["fecha_periodo"] = pd.to_datetime(summary["periodo"] + "-01")
summary["pct_mes_anterior"] = summary.groupby("sexo")["precio_promedio"].pct_change() * 100
summary["pct_mismo_mes_anio_anterior"] = summary.groupby(["sexo","mes"])["precio_promedio"].pct_change() * 100
summary["direccion"] = summary["pct_mes_anterior"].apply(
    lambda x: "subio" if pd.notna(x) and x > 0 else ("bajo" if pd.notna(x) and x < 0 else "sin dato")
)

display_cols = ["sexo","periodo","precio_promedio","pct_mes_anterior","direccion","pct_mismo_mes_anio_anterior"]
summary[display_cols].round(1)

In [ ]:
# Celda 8 - Grafico interactivo con selector de sexo
sexos = sorted(summary["sexo"].unique())
sexo_sel = sexos[0]  # cambia aqui o usa el menu del grafico

sub = summary[summary["sexo"] == sexo_sel]
colors = ["#2ca02c" if (v or 0) >= 0 else "#d62728" for v in sub["pct_mes_anterior"].fillna(0)]

fig = make_subplots(rows=1, cols=2, subplot_titles=(f"Precio {sexo_sel}", f"Variacion % {sexo_sel}"))
fig.add_trace(go.Scatter(x=sub["fecha_periodo"], y=sub["precio_promedio"], mode="lines+markers", name="Precio"), row=1, col=1)
fig.add_trace(go.Bar(x=sub["periodo"], y=sub["pct_mes_anterior"], marker_color=colors, name="% cambio"), row=1, col=2)
fig.add_hline(y=0, line_dash="dash", row=1, col=2)
fig.update_layout(height=450, template="plotly_white", title=f"Precio por sexo: {sexo_sel}")
fig.show()

In [ ]:
# Celda 9 - Comparar mismo mes entre anos (por sexo)
sexo_sel = "ML"  # elige sexo
sub = summary[summary["sexo"] == sexo_sel]
fig = go.Figure()
for year in sorted(sub["anio"].unique()):
    ysub = sub[sub["anio"] == year]
    fig.add_trace(go.Bar(x=ysub["mes_nombre"], y=ysub["precio_promedio"], name=str(int(year))))
fig.update_layout(barmode="group", title=f"Mismo mes entre anos - {sexo_sel}", template="plotly_white", height=450)
fig.show()

## 6. Guardar resultados

Los archivos quedan en **`MyDrive/subasta_ganadera_data/`** (`data/lotes/`, `data/resumen/`, `export/`).
Tambien puedes descargar el CSV:

In [ ]:
# Celda 10 - Descargar CSV
from google.colab import files
files.download(str(EXPORT_DIR / "lotes_consolidado.csv"))